# Random Guide-Selection Strategies

Compares different random strategies for selecting guide sets.
Each strategy gets its own run directory matched by name pattern.

### TODOs
- Add best single guide comparison (k=1 vs k>1)
- Revisit metrics — consider adding new ones
- Egraph growth curves from `top_k.json` per-iteration data (nodes/classes over iterations, averaged across trials, by k) -> File is very big

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from helpers import find_latest_run, load_top_k

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 8)})

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
# Add new strategies here: (display_name, run-directory pattern)
STRATEGIES = [
    ("full-union", "random-fullunion-true"),
    ("root-union", "random-fullunion-false"),
]

In [ ]:
# ── Load & parse ──────────────────────────────────────────────────────
frames = []
for name, pattern in STRATEGIES:
    run_dir = find_latest_run(pattern)
    print(f"{name}: {run_dir}")
    frame = load_top_k(run_dir, name)
    print(f"  {len(frame)} trial rows")
    frames.append(frame)

df = pl.concat(frames).with_columns(
    (pl.col("nodes") / pl.col("classes")).alias("nodes_per_class"),
)
k_values = sorted(df["k"].unique().to_list())
strat_names = [s[0] for s in STRATEGIES]

print(f"\nTotal rows: {len(df)}, k values: {k_values}")
df.head(5)

## Metrics vs k

In [ ]:
PALETTE = plt.cm.tab10.colors  # type: ignore[attr-defined]
MARKER_CYCLE = ["o", "s", "D", "^", "v", "P", "X", "*"]


def strat_style(i):
    return {
        "color": PALETTE[i % len(PALETTE)],
        "marker": MARKER_CYCLE[i % len(MARKER_CYCLE)],
    }


METRICS = ["iters", "nodes", "classes", "nodes_per_class", "total_time"]
n_metrics = len(METRICS)
fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 5))

for metric, ax in zip(METRICS, axes):
    for i, strat in enumerate(strat_names):
        style = strat_style(i)
        agg = (
            df.filter(pl.col("strategy") == strat)
            .group_by("k")
            .agg(
                pl.col(metric).mean().alias("mean"),
                pl.col(metric).std().alias("std"),
            )
            .sort("k")
        )
        ks = agg["k"].to_numpy()
        means = agg["mean"].to_numpy()
        stds = agg["std"].fill_null(0).to_numpy()
        ax.plot(ks, means, label=strat, **style)
        ax.fill_between(
            ks, means - stds, means + stds, alpha=0.15, color=style["color"]
        )

    ax.set_xlabel("k (number of guides)")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} vs k (mean \u00b1 std)")
    ax.legend()
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.set_xticks(k_values)

fig.tight_layout()
plt.show()

## Reachability rate vs k

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for i, strat in enumerate(strat_names):
    style = strat_style(i)
    reach = (
        df.filter(pl.col("strategy") == strat)
        .group_by("k")
        .agg(pl.col("iters").is_not_null().mean().alias("reach_rate"))
        .sort("k")
    )
    ax.plot(
        reach["k"].to_numpy(),
        reach["reach_rate"].to_numpy() * 100,
        label=strat,
        **style,
    )

ax.set_xlabel("k (number of guides)")
ax.set_ylabel("reachability (%)")
ax.set_title("Fraction of trials reaching the goal vs k")
ax.legend()
ax.set_xscale("log")
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.set_xticks(k_values)
ax.set_ylim(-5, 105)
fig.tight_layout()
plt.show()

## Box plots: iterations by strategy at each k

In [ ]:
n_k = len(k_values)
fig, axes = plt.subplots(1, n_k, figsize=(4 * n_k, 5), squeeze=False, sharey=True)

for ki, k in enumerate(k_values):
    ax = axes[0][ki]
    box_data = []
    labels = []
    colors = []
    for i, strat in enumerate(strat_names):
        vals = (
            df.filter((pl.col("k") == k) & (pl.col("strategy") == strat))["iters"]
            .drop_nulls()
            .to_numpy()
        )
        if len(vals) > 0:
            box_data.append(vals)
            labels.append(strat)
            colors.append(PALETTE[i % len(PALETTE)])
    if box_data:
        bp = ax.boxplot(box_data, tick_labels=labels, patch_artist=True)
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
    ax.set_title(f"k = {k}")
    ax.set_ylabel("iters" if ki == 0 else "")

fig.suptitle("Iterations by strategy at each k", fontsize=13)
fig.tight_layout()
plt.show()

## Box plots: nodes by strategy at each k

In [ ]:
fig, axes = plt.subplots(1, n_k, figsize=(4 * n_k, 5), squeeze=False, sharey=True)

for ki, k in enumerate(k_values):
    ax = axes[0][ki]
    box_data = []
    labels = []
    colors = []
    for i, strat in enumerate(strat_names):
        vals = (
            df.filter((pl.col("k") == k) & (pl.col("strategy") == strat))["nodes"]
            .drop_nulls()
            .to_numpy()
        )
        if len(vals) > 0:
            box_data.append(vals)
            labels.append(strat)
            colors.append(PALETTE[i % len(PALETTE)])
    if box_data:
        bp = ax.boxplot(box_data, tick_labels=labels, patch_artist=True)
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
    ax.set_title(f"k = {k}")
    ax.set_ylabel("nodes" if ki == 0 else "")

fig.suptitle("Nodes by strategy at each k", fontsize=13)
fig.tight_layout()
plt.show()

## Summary table

In [ ]:
summary = (
    df.group_by("strategy", "k")
    .agg(
        pl.col("iters").mean().round(2).alias("avg_iters"),
        pl.col("nodes").mean().round(0).alias("avg_nodes"),
        pl.col("classes").mean().round(0).alias("avg_classes"),
        pl.col("nodes_per_class").mean().round(2).alias("avg_nodes_per_class"),
        pl.col("total_time").mean().round(4).alias("avg_time_s"),
        pl.col("iters").is_not_null().mean().round(3).alias("reachability"),
        pl.col("iters").count().alias("n_trials"),
    )
    .sort("k", "strategy")
)
summary

## Per-goal analysis

In [ ]:
# Per-goal reachability at each k (shared by several plots below)
goal_reach = (
    df.group_by("goal", "k")
    .agg(pl.col("iters").is_not_null().mean().alias("reach_rate"))
    .sort("goal", "k")
)

# Overall difficulty: reachability averaged across all k values
goal_difficulty = (
    goal_reach.group_by("goal")
    .agg(pl.col("reach_rate").mean().alias("avg_reach"))
    .sort("avg_reach")
)

# Pivot to wide form: one column per k
goal_reach_wide = goal_reach.pivot(on="k", index="goal", values="reach_rate").sort(
    "goal"
)

print(
    f"{goal_reach.shape[0]} goal\u00d7k combinations, {goal_difficulty.shape[0]} unique goals"
)

## Goal difficulty distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of per-goal avg reachability
ax = axes[0]
avg_reach = goal_difficulty["avg_reach"].to_numpy()
ax.hist(avg_reach, bins=20, edgecolor="black", alpha=0.7)
ax.axvline(
    np.median(avg_reach),
    color="red",
    ls="--",
    label=f"median={np.median(avg_reach):.2f}",
)
ax.set_xlabel("Average reachability (across all k)")
ax.set_ylabel("Number of goals")
ax.set_title("Goal difficulty distribution")
ax.legend()

# CDF per k
ax = axes[1]
for i, k in enumerate(k_values):
    reach_at_k = (
        goal_reach.filter(pl.col("k") == k).sort("reach_rate")["reach_rate"].to_numpy()
    )
    ax.plot(
        reach_at_k,
        np.linspace(0, 1, len(reach_at_k)),
        label=f"k={k}",
        color=PALETTE[i % len(PALETTE)],
    )
ax.set_xlabel("Per-goal reachability rate")
ax.set_ylabel("Fraction of goals (CDF)")
ax.set_title("CDF of per-goal reachability by k")
ax.legend()

fig.tight_layout()
plt.show()

## Goal-level reachability heatmap

In [ ]:
# Sort goals by average reachability (hardest at top)
goal_order = goal_difficulty["goal"].to_list()

# Build matrix: rows = goals (sorted by difficulty), columns = k values
matrix = np.full((len(goal_order), len(k_values)), np.nan)
goal_idx = {g: i for i, g in enumerate(goal_order)}
for row in goal_reach.iter_rows(named=True):
    gi = goal_idx[row["goal"]]
    ki = k_values.index(row["k"])
    matrix[gi, ki] = row["reach_rate"]

fig, ax = plt.subplots(figsize=(8, 12))
im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(len(k_values)))
ax.set_xticklabels([str(k) for k in k_values])
ax.set_xlabel("k (number of guides)")
ax.set_ylabel("Goal (sorted by avg reachability, hardest at top)")
ax.set_yticks([])  # too many goals for individual labels
ax.set_title("Per-goal reachability rate by k")
fig.colorbar(im, ax=ax, label="Reachability rate", shrink=0.6)
fig.tight_layout()
plt.show()

## Diminishing returns: reachability improvement per goal

In [ ]:
# For each goal, plot reachability vs k and color by difficulty cluster
# Cluster: easy (reach@k=1 > 0.7), hard (reach@k=100 < 0.3), scalable (rest)
reach_at_k1 = (
    goal_reach.filter(pl.col("k") == k_values[0])
    .select("goal", "reach_rate")
    .rename({"reach_rate": "reach_k1"})
)
reach_at_kmax = (
    goal_reach.filter(pl.col("k") == k_values[-1])
    .select("goal", "reach_rate")
    .rename({"reach_rate": "reach_kmax"})
)
goal_clusters = reach_at_k1.join(reach_at_kmax, on="goal").with_columns(
    pl.when(pl.col("reach_k1") > 0.7)
    .then(pl.lit("easy"))
    .when(pl.col("reach_kmax") < 0.3)
    .then(pl.lit("hard"))
    .otherwise(pl.lit("scalable"))
    .alias("cluster")
)

cluster_colors = {"easy": "green", "scalable": "steelblue", "hard": "red"}
cluster_counts = goal_clusters.group_by("cluster").len().sort("cluster")
print("Goal clusters:")
print(cluster_counts)

fig, ax = plt.subplots(figsize=(10, 6))
for cluster, color in cluster_colors.items():
    goals_in = goal_clusters.filter(pl.col("cluster") == cluster)["goal"].to_list()
    for gi, g in enumerate(goals_in):
        row = goal_reach.filter(pl.col("goal") == g).sort("k")
        ax.plot(
            row["k"].to_numpy(),
            row["reach_rate"].to_numpy() * 100,
            color=color,
            alpha=0.4,
            linewidth=1,
            label=cluster if gi == 0 else None,
        )

ax.set_xlabel("k (number of guides)")
ax.set_ylabel("Reachability (%)")
ax.set_title(
    f"Per-goal reachability vs k (easy: reach@k={k_values[0]}>70%, hard: reach@k={k_values[-1]}<30%)"
)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.set_xticks(k_values)
ax.set_ylim(-5, 105)
ax.legend()
fig.tight_layout()
plt.show()

## Nodes vs reachability tradeoff

In [ ]:
# Per strategy×k: avg nodes (reached only) vs reachability
tradeoff = (
    df.group_by("strategy", "k")
    .agg(
        pl.col("iters").is_not_null().mean().alias("reachability"),
        pl.col("nodes").mean().alias("avg_nodes"),
        pl.col("total_time").mean().alias("avg_time"),
    )
    .sort("strategy", "k")
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, strat in enumerate(strat_names):
    style = strat_style(i)
    sub = tradeoff.filter(pl.col("strategy") == strat)
    ks = sub["k"].to_list()

    # Nodes vs reachability
    ax = axes[0]
    ax.plot(
        sub["avg_nodes"].to_numpy(),
        sub["reachability"].to_numpy() * 100,
        **style,
        label=strat,
    )
    for k_val, x, y in zip(
        ks, sub["avg_nodes"].to_list(), sub["reachability"].to_list()
    ):
        ax.annotate(
            f"k={k_val}",
            (x, y * 100),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=7,
        )

    # Time vs reachability
    ax = axes[1]
    ax.plot(
        sub["avg_time"].to_numpy() * 1000,
        sub["reachability"].to_numpy() * 100,
        **style,
        label=strat,
    )
    for k_val, x, y in zip(
        ks, sub["avg_time"].to_list(), sub["reachability"].to_list()
    ):
        ax.annotate(
            f"k={k_val}",
            (x * 1000, y * 100),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=7,
        )

axes[0].set_xlabel("Avg nodes (all trials)")
axes[0].set_ylabel("Reachability (%)")
axes[0].set_title("Reachability vs node cost")
axes[0].legend()

axes[1].set_xlabel("Avg time (ms)")
axes[1].set_ylabel("Reachability (%)")
axes[1].set_title("Reachability vs time cost")
axes[1].legend()

fig.tight_layout()
plt.show()